# MiniGenie — VQ-VAE Training

Train the VQ-VAE tokenizer on CoinRun episode frames.  
**Spec:** `docs/build_spec.md` §2.2 — 512×256 codebook, EMA updates, dead code reset.  
**Config:** `configs/vqvae.yaml` — 50K steps, batch 64, lr 3e-4, cosine schedule.

### Targets
- Reconstruction PSNR > 28 dB
- Codebook utilization > 80%

### Expected runtime
- T4: ~4–6 hours for 50K steps
- A100: ~1–2 hours for 50K steps

---
## 1. Setup (mount Drive, clone code, install deps, verify GPU)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT = '/content/drive/MyDrive/minigenie'
os.makedirs(f'{DRIVE_PROJECT}/checkpoints/vqvae', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/samples_vqvae', exist_ok=True)
print(f'Drive project root: {DRIVE_PROJECT}')

In [ ]:
REPO_URL = 'https://github.com/BrutalCaeser/minigenie.git'  # <-- UPDATE THIS
LOCAL_CODE = '/content/minigenie'

if os.path.exists(LOCAL_CODE):
    !cd {LOCAL_CODE} && git pull --ff-only
else:
    !git clone {REPO_URL} {LOCAL_CODE}

os.chdir(LOCAL_CODE)
!git log --oneline -3

In [ ]:
!pip install -q einops pyyaml wandb tqdm imageio scipy pillow matplotlib torchvision
!pip install -q -e .

import torch
assert torch.cuda.is_available(), 'No GPU! Go to Runtime > Change runtime type > GPU'
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')

---
## 2. Extract & Verify Data

In [ ]:
import glob
import numpy as np

DATA_TAR = f'{DRIVE_PROJECT}/coinrun_data.tar.gz'
LOCAL_DATA = '/content/minigenie/data/coinrun/episodes'

if os.path.exists(LOCAL_DATA) and len(os.listdir(LOCAL_DATA)) > 100:
    print('Data already extracted.')
elif os.path.exists(DATA_TAR):
    print('Extracting data...')
    !tar xzf {DATA_TAR} -C /content/minigenie/
else:
    raise FileNotFoundError(
        f'Data archive not found at {DATA_TAR}\n'
        'Upload coinrun_data.tar.gz to Google Drive at MyDrive/minigenie/'
    )

# Filter out Apple Double ._* files
paths = sorted([p for p in glob.glob(f'{LOCAL_DATA}/*.npz')
                if not os.path.basename(p).startswith('._')])
print(f'Episodes: {len(paths)}')

# Spot check
ep = np.load(paths[0])
print(f'Sample: frames {ep["frames"].shape} {ep["frames"].dtype}, '
      f'actions {ep["actions"].shape} range [{ep["actions"].min()}, {ep["actions"].max()}]')
assert len(paths) >= 1000, f'Expected >=1000 episodes, got {len(paths)}'
print('Data OK')

---
## 3. Symlink Checkpoints to Google Drive

Checkpoints are saved to Drive so they persist across Colab disconnections.  
The training script writes to `checkpoints/vqvae/` — we symlink this to Drive.

In [ ]:
import shutil

# Symlink local checkpoint dir -> Drive (persists across disconnects)
LOCAL_CKPT = '/content/minigenie/checkpoints/vqvae'
DRIVE_CKPT = f'{DRIVE_PROJECT}/checkpoints/vqvae'

os.makedirs(os.path.dirname(LOCAL_CKPT), exist_ok=True)
if os.path.islink(LOCAL_CKPT):
    os.remove(LOCAL_CKPT)
elif os.path.isdir(LOCAL_CKPT):
    shutil.rmtree(LOCAL_CKPT)
os.symlink(DRIVE_CKPT, LOCAL_CKPT)

# Also symlink samples dir
LOCAL_SAMPLES = '/content/minigenie/samples_vqvae'
DRIVE_SAMPLES = f'{DRIVE_PROJECT}/samples_vqvae'
os.makedirs(DRIVE_SAMPLES, exist_ok=True)
if os.path.islink(LOCAL_SAMPLES):
    os.remove(LOCAL_SAMPLES)
elif os.path.isdir(LOCAL_SAMPLES):
    shutil.rmtree(LOCAL_SAMPLES)
os.symlink(DRIVE_SAMPLES, LOCAL_SAMPLES)

existing = sorted(glob.glob(f'{DRIVE_CKPT}/step_*.pt'))
if existing:
    print(f'Found {len(existing)} existing checkpoint(s): {[os.path.basename(p) for p in existing]}')
    print('Training will auto-resume from the latest.')
else:
    print('No existing checkpoints. Training will start from scratch.')
print(f'\nCheckpoints -> {DRIVE_CKPT}')

---
## 4. Run Smoke Test

Quick end-to-end verification with random data before committing to a long training run.

In [ ]:
!cd /content/minigenie && python scripts/smoke_test.py

---
## 5. Train VQ-VAE

This calls the full training loop from `src/training/train_vqvae.py`.  
Training resumes automatically from the latest checkpoint on Drive.

**Monitor:**
- Loss should decrease steadily
- Codebook utilization should reach >80% by step 10K
- If utilization <50% at step 5K: stop, increase reset frequency, restart

**Override defaults** by passing keyword arguments to `train()` below.

In [ ]:
from src.training.train_vqvae import train

train(
    data_dir='/content/minigenie/data/coinrun/episodes',
    ckpt_dir='/content/minigenie/checkpoints/vqvae',
    config_path='/content/minigenie/configs/vqvae.yaml',
    # --- Override any config values here ---
    # max_steps=50000,
    # batch_size=64,
    # lr=3e-4,
    resume=True,
    device='cuda',
)

---
## 6. Inspect Results

After training completes (or after interruption), inspect reconstructions and metrics.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Display saved reconstruction samples
samples_dir = f'{DRIVE_PROJECT}/samples_vqvae'
sample_files = sorted(glob.glob(f'{samples_dir}/step_*.png'))

if sample_files:
    print(f'Found {len(sample_files)} sample images')
    # Show the last 3
    for path in sample_files[-3:]:
        img = Image.open(path)
        fig, ax = plt.subplots(1, 1, figsize=(12, 6))
        ax.imshow(img)
        ax.set_title(os.path.basename(path))
        ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print('No sample images found yet. Train for at least sample_every steps.')

### Compute PSNR on a Batch

In [ ]:
import torch
import numpy as np
from src.models.vqvae import VQVAE
from src.training.checkpoint import CheckpointManager
from src.training.train_vqvae import FrameDataset
from torch.utils.data import DataLoader

# Load latest checkpoint
ckpt_mgr = CheckpointManager('/content/minigenie/checkpoints/vqvae')
state = ckpt_mgr.load_latest()

if state is not None:
    import yaml
    with open('/content/minigenie/configs/vqvae.yaml') as f:
        cfg = yaml.safe_load(f)
    mcfg = cfg['model']

    model = VQVAE(
        in_channels=mcfg.get('in_channels', 3),
        hidden_channels=mcfg.get('hidden_channels', [64, 128, 256]),
        codebook_size=mcfg.get('codebook_size', 512),
        embed_dim=mcfg.get('embed_dim', 256),
        num_res_blocks=mcfg.get('num_res_blocks', 2),
        ema_decay=mcfg.get('ema_decay', 0.99),
        commitment_cost=mcfg.get('commitment_cost', 0.25),
    ).cuda()
    model.load_state_dict(state['model'])
    model.eval()
    step = state['step']
    print(f'Loaded checkpoint at step {step}')

    # Compute PSNR over ~1000 frames
    dataset = FrameDataset('/content/minigenie/data/coinrun/episodes', resolution=64)
    loader = DataLoader(dataset, batch_size=64, shuffle=True)

    psnr_values = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= 16:  # 16 x 64 = 1024 frames
                break
            x = batch.cuda()
            x_hat, _, _, _ = model(x)
            # Per-image MSE -> PSNR
            mse = ((x - x_hat) ** 2).mean(dim=(1, 2, 3))
            psnr = -10 * torch.log10(mse + 1e-8)
            psnr_values.extend(psnr.cpu().tolist())

    psnr_arr = np.array(psnr_values)
    print(f'\nPSNR over {len(psnr_arr)} frames:')
    print(f'   Mean:   {psnr_arr.mean():.2f} dB')
    print(f'   Median: {np.median(psnr_arr):.2f} dB')
    print(f'   Min:    {psnr_arr.min():.2f} dB')
    print(f'   Max:    {psnr_arr.max():.2f} dB')
    target_psnr = cfg.get('targets', {}).get('psnr_db', 28.0)
    status = 'PASS' if psnr_arr.mean() >= target_psnr else 'BELOW TARGET'
    print(f'   Target: {target_psnr} dB [{status}]')

    # Codebook utilization
    all_indices = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= 16:
                break
            z_e = model.encoder(batch.cuda())
            _, indices, _ = model.quantizer(z_e)
            all_indices.append(indices.cpu())
    all_indices = torch.cat(all_indices).flatten()
    unique = all_indices.unique().numel()
    total = model.quantizer.codebook.shape[0]
    util = 100.0 * unique / total
    target_util = cfg.get('targets', {}).get('codebook_utilization_pct', 80.0)
    status = 'PASS' if util >= target_util else 'BELOW TARGET'
    print(f'\nCodebook utilization: {unique}/{total} = {util:.1f}% (target: {target_util}%) [{status}]')
else:
    print('No checkpoint found. Train the model first.')

---
## 7. Post-Session Checklist

After training completes or before Colab disconnects:

1. Checkpoints are already on Drive (symlinked)
2. Sample reconstructions are already on Drive (symlinked)
3. Update `logs/TRAINING_LOG.md` with session results
4. Record: final loss, PSNR, codebook utilization, GPU type, steps completed
5. Push any code changes to GitHub

In [ ]:
# Summary of what's on Drive
print('=== Drive Contents ===')
for subdir in ['checkpoints/vqvae', 'samples_vqvae']:
    full = f'{DRIVE_PROJECT}/{subdir}'
    if os.path.exists(full):
        files = os.listdir(full)
        print(f'  {subdir}/: {len(files)} files')
        for f in sorted(files)[-5:]:
            size_mb = os.path.getsize(os.path.join(full, f)) / 1e6
            print(f'    {f} ({size_mb:.1f} MB)')
    else:
        print(f'  {subdir}/: (not found)')